customer_support_system/
- .mcp.json # MCP server registry
- .claude/settings.json # settings.json
- README.md
- requirements.txt
- .env.example
- .gitignore
- main.py # Entry point — runs a query through the system
- jumpstarter.py # Jumpstarter pattern — shared session state
- models.py # Pydantic models for typed results
- coordinator.py # Coordinator agent — decomposes query, dispatches
- hitl.py # Minimal HITL helper (orders agent only)ç

- subagents/
-   __init__.py
-   base.py  # Common subagent scaffolding (lifecycle hooks)
-   orders.py # Sub-agent 1 — order lookups (uses internal MCP)
-   policy.py # Sub-agent 2 — policy lookup (uses grep tool)
-   crm.py # Sub-agent 3 — CRM lookup (uses external MCP)
-   summarizer.py # Summarizer agent — aggregates the 3 outputs

- mcp_servers/
-   __init__.py
-   internal_mcp.py # Internal MCP server built with FastMCP
-   external_mcp_config.py # External MCP — config only (server is 3rd-party)

- tools
-   __init__.py
-    internal_tools.py            # Native Python tools — grep + web_search

- docs/return_policy.md # Sample policy doc grep'd by policy agent

In [ ]:
# File 1 · requirements.txt
# Claude Agent SDK — coordinator and subagents
anthropic>=0.40.0
claude-agent-sdk>=0.2.0

# FastMCP — minimal framework for building internal MCP servers
fastmcp>=0.3.0

# MCP client — to consume both internal and external MCP servers
mcp>=1.0.0

# Tooling
pydantic>=2.0.0          # Typed result envelopes
python-dotenv>=1.0.0     # Environment variables
httpx>=0.27.0            # HTTP client for tools/web search
rich>=13.0.0             # Pretty CLI output


In [ ]:
# File 2 · .env.example
# ── Anthropic API ──
ANTHROPIC_API_KEY=sk-ant-api03-...

# ── External MCP server (vendor-hosted CRM connector) ──
EXTERNAL_MCP_URL=https://crm.example.com/mcp
EXTERNAL_MCP_API_KEY=ext-mcp-key-...

# ── Brave / Tavily / SerpAPI for the web_search tool ──
SEARCH_API_KEY=

# ── Confidence threshold for human escalation ──
ESCALATION_THRESHOLD=0.7

# File 3 · .gitignore
# Secrets
.env
.env.*.local

# Local Claude settings (personal overrides)
.claude/settings.local.json
.claude/projects/

# Python
__pycache__/
*.pyc
.venv/
venv/
.pytest_cache/
*.egg-info/

In [ ]:
# File 4 · .mcp.json
{
  "mcpServers": {
    "internal-customer-data": {
      "//": "Our FastMCP server — runs as a local subprocess via stdio",
      "command": "python",
      "args": ["-m", "mcp_servers.internal_mcp"]
    },

    "external-crm": {
      "//": "Vendor-hosted MCP server reached over HTTP",
      "type": "http",
      "url": "${EXTERNAL_MCP_URL}",
      "headers": {
        "Authorization": "Bearer ${EXTERNAL_MCP_API_KEY}"
      }
    }
  }
}

In [ ]:
# File 5 · .claude/settings.json
{
  "//": "Project-scoped settings for Claude Code / Agent SDK in this repo.",

  "mcpServers": {
    "enabled": ["internal-customer-data", "external-crm"]
  },

  "permissions": {
    "allow": [
      "Read",
      "Edit(./subagents/**)",
      "Edit(./tools/**)",
      "Bash(python *)",
      "Bash(pytest *)",
      "mcp__internal-customer-data__search_orders",
      "mcp__internal-customer-data__get_customer_profile",
      "mcp__external-crm__crm_get_account"
    ],
    "ask": [
      "//": "Framework-level HITL — these tools always prompt the user.",
      "mcp__internal-customer-data__create_return_request",
      "mcp__internal-customer-data__cancel_order"
    ],
    "deny": [
      "Bash(rm -rf *)",
      "Bash(git push origin main)"
    ]
  },

  "model": "claude-sonnet-4-5",

  "env": {
    "ESCALATION_THRESHOLD": "0.7",
    "PYTHONUNBUFFERED": "1"
  }
}

In [ ]:
# File 6 · models.py
"""Typed Pydantic models for inter-agent communication.

Why this matters (cert-relevant):
- Subagents communicate via structured JSON, not free-form text.
- The coordinator inspects `status` to decide what happens next:
    SUCCESS  → pass result to next stage
    ERROR    → retry with backoff
    ESCALATE → hand off to human / summarizer
- This is the "structured context passing between agents" pattern.
"""
from enum import Enum
from typing import Any, Literal, Optional
from pydantic import BaseModel, Field


class AgentStatus(str, Enum):
    """The three terminal states a subagent can return."""
    SUCCESS = "success"      # Subagent completed, data is in `data`
    ERROR = "error"          # Transient/recoverable failure → coordinator can retry
    ESCALATE = "escalate"    # Needs human — confidence low or sensitive topic


class SubAgentResult(BaseModel):
    """Standard envelope every subagent returns to the coordinator."""
    agent_name: str = Field(..., description="Which subagent produced this result")
    status: AgentStatus
    data: Optional[dict[str, Any]] = Field(
        default=None,
        description="Payload — only meaningful if status == SUCCESS"
    )
    confidence: float = Field(
        default=1.0, ge=0.0, le=1.0,
        description="Subagent's confidence in its answer (0-1)"
    )
    error_message: Optional[str] = Field(
        default=None,
        description="Human-readable error — only present if status == ERROR"
    )
    escalation_reason: Optional[str] = Field(
        default=None,
        description="Why we need a human — only present if status == ESCALATE"
    )
    tools_used: list[str] = Field(
        default_factory=list,
        description="Audit trail — which tools/MCPs were invoked"
    )


class TaskAssignment(BaseModel):
    """One unit of work the coordinator dispatches to a subagent."""
    target_agent: Literal["orders", "policy", "crm"]
    instruction: str = Field(..., description="Natural-language task for the subagent")
    parameters: dict[str, Any] = Field(default_factory=dict)


class FinalAnswer(BaseModel):
    """What the summarizer returns to the user."""
    answer: str
    sources: list[str]
    needs_human_followup: bool
    followup_reason: Optional[str] = None


In [ ]:
# File 7 · hitl.py
"""Minimal Human-in-the-Loop helper.

Used only by the orders subagent. If a tool call is sensitive, we pause
execution, ask a human via CLI, and proceed with their decision.

Cert-relevant pattern: high-consequence financial actions (refunds,
cancellations) require human approval — this is the canonical "human
escalation triggers" example from chapter 10.
"""

# Tool names that require human approval before execution.
# Anything not in this set runs normally without HITL interference.
SENSITIVE_TOOLS = {"create_return_request", "process_refund", "cancel_order"}


def needs_human_approval(tool_name: str) -> bool:
    """Returns True if this tool call should pause for human approval."""
    return tool_name in SENSITIVE_TOOLS


def ask_human(tool_name: str, args: dict) -> bool:
    """Pause execution and ask the human via CLI prompt.

    Returns:
        True  if human approves → proceed with the tool call
        False if human denies   → skip the tool call

    In production, replace the input() call with:
      - Slack message + button click
      - Dashboard webhook
      - Queue-based async review
    The interface (returning a bool) stays the same.
    """
    print("\n" + "=" * 60)
    print("  🔔 HUMAN APPROVAL NEEDED")
    print("=" * 60)
    print(f"  Tool:      {tool_name}")
    print(f"  Arguments: {args}")
    print("-" * 60)
    answer = input("  Approve? [y/N]: ").strip().lower()
    return answer == "y"

In [ ]:
# File 8 · jumpstarter.py
"""Jumpstarter pattern — centralized state bootstrap for the agent session.

Purpose:
- Initializes shared state ONCE at session start (via on_start hook).
- Passes itself by reference into every subagent so they read/write the SAME state.
- Persists across the full coordinator → subagent → summarizer flow.
- Provides audit logging, MCP client pool, and config in one place.
"""
import os
import time
import uuid
from typing import Any, Optional
from dotenv import load_dotenv

from mcp_servers.internal_mcp import get_internal_mcp_client
from mcp_servers.external_mcp_config import get_external_mcp_client


class Jumpstarter:
    """Shared session state — created once per user query, passed everywhere."""

    def __init__(self, user_query: str, user_id: Optional[str] = None):
        load_dotenv()

        # ── Identity & request metadata ──
        self.session_id: str = str(uuid.uuid4())
        self.user_id: str = user_id or "anonymous"
        self.user_query: str = user_query
        self.started_at: float = time.time()

        # ── Configuration loaded from environment ──
        self.anthropic_api_key: str = os.environ["ANTHROPIC_API_KEY"]
        self.escalation_threshold: float = float(
            os.environ.get("ESCALATION_THRESHOLD", "0.7")
        )

        # ── MCP clients (populated by bootstrap()) ──
        self.internal_mcp = None
        self.external_mcp = None

        # ── Shared mutable state across agents ──
        self.scratchpad: dict[str, Any] = {}

        # ── Audit trail — every tool call, retry, HITL event ──
        self.audit_log: list[dict[str, Any]] = []

        # ── Retry counters per subagent ──
        self.retry_counts: dict[str, int] = {"orders": 0, "policy": 0, "crm": 0}

    # ──────────────────────────────────────────────────────────
    # Lifecycle methods (mapped to Agent SDK hooks in production)
    # ──────────────────────────────────────────────────────────
    def bootstrap(self) -> None:
        """Open MCP connections and warm up resources. on_start equivalent."""
        self.log_event("bootstrap_start", {})
        self.internal_mcp = get_internal_mcp_client()
        self.external_mcp = get_external_mcp_client()
        self.log_event("bootstrap_complete", {"session_id": self.session_id})

    def shutdown(self) -> None:
        """Close MCP connections and flush logs. on_end equivalent."""
        if self.external_mcp:
            self.external_mcp.close()
        elapsed = time.time() - self.started_at
        self.log_event("shutdown", {"elapsed_seconds": round(elapsed, 2)})

    # ──────────────────────────────────────────────────────────
    # Audit log helpers
    # ──────────────────────────────────────────────────────────
    def log_event(self, kind: str, details: dict[str, Any]) -> None:
        self.audit_log.append({
            "ts": time.time(),
            "session_id": self.session_id,
            "kind": kind,
            "details": details,
        })

    def increment_retry(self, agent_name: str) -> int:
        self.retry_counts[agent_name] += 1
        return self.retry_counts[agent_name]

In [ ]:
# File 9 · mcp_servers/__init__.py
# Empty — marks mcp_servers/ as a Python package

In [ ]:
# File 10 · mcp_servers/internal_mcp.py
"""Internal MCP server built with FastMCP.

Exposes our company's internal data — orders, returns, customer profiles —
as MCP Tools and Resources. The orders subagent calls this.

Cert-relevant distinction:
- Tools (state-querying actions): search_orders, create_return_request
- Resources (read-only reference): order_schema
"""
from fastmcp import FastMCP

mcp = FastMCP("internal-customer-data")


# ──────────────────────────────────────────────────────────
# MCP TOOLS — actions that query or modify state
# ──────────────────────────────────────────────────────────

@mcp.tool()
def search_orders(order_id: str) -> dict:
    """Look up an order by ID in the internal order management system.

    Returns order details or {"error": "NOT_FOUND"} if the order doesn't exist.
    """
    fake_orders = {
        "ORD-12345": {
            "order_id": "ORD-12345",
            "status": "shipped",
            "tracking": "1Z999AA10123456784",
            "items": [{"sku": "PHONE-X", "qty": 1, "price": 899.00}],
            "total": 899.00,
            "customer_id": "CUST-9876",
        },
        "ORD-99999": {
            "order_id": "ORD-99999",
            "status": "delivered",
            "items": [{"sku": "BOOK-A", "qty": 2, "price": 19.99}],
            "total": 39.98,
            "customer_id": "CUST-9876",
        },
    }
    if order_id not in fake_orders:
        return {"error": "NOT_FOUND", "message": f"Order {order_id} does not exist"}
    return fake_orders[order_id]


@mcp.tool()
def get_customer_profile(customer_id: str) -> dict:
    """Fetch a customer's profile from the internal CRM cache."""
    return {
        "customer_id": customer_id,
        "name": "Maria González",
        "tier": "gold",
        "lifetime_value": 4520.00,
        "open_tickets": 0,
    }


@mcp.tool()
def create_return_request(order_id: str, reason: str) -> dict:
    """Create a return request in the OMS.

    Side-effect tool — modifies state. SENSITIVE → goes through HITL.
    """
    return {
        "return_id": f"RET-{order_id[-5:]}",
        "status": "pending_review",
        "estimated_refund_days": 7,
    }


# ──────────────────────────────────────────────────────────
# MCP RESOURCES — read-only reference material
# ──────────────────────────────────────────────────────────

@mcp.resource("schema://order")
def order_schema() -> str:
    """Order schema definition. Resources don't change state — read-only."""
    return """Order schema:
    - order_id: string (format ORD-XXXXX)
    - status: enum(pending, shipped, delivered, cancelled, returned)
    - tracking: string | null
    - items: array of {sku, qty, price}
    - total: float
    - customer_id: string
    """


# ──────────────────────────────────────────────────────────
# Client factory + standalone server entry
# ──────────────────────────────────────────────────────────

class _InProcessClient:
    """Simple in-process wrapper so the demo runs without spawning a subprocess.

    In production, the FastMCP server runs as a separate process and the
    client connects via stdio (declared in .mcp.json).
    """
    def __init__(self, server: FastMCP):
        self._server = server

    def call_tool_sync(self, tool_name: str, args: dict) -> dict:
        # FastMCP exposes tools as functions — fetch and invoke directly.
        # Replace with `mcp.client.call_tool()` when running over stdio/http.
        tool_fn = self._server._tools[tool_name].fn
        return tool_fn(**args)


def get_internal_mcp_client():
    """Returns a connected MCP client to this server."""
    return _InProcessClient(mcp)


# Standalone server entry: `python -m mcp_servers.internal_mcp`
if __name__ == "__main__":
    mcp.run()

In [ ]:
# File 11 · mcp_servers/external_mcp_config.py
"""External MCP server configuration.

The "external MCP" is a 3rd-party MCP server we DON'T run — it's hosted by
a vendor (Salesforce / HubSpot / Zendesk). We connect to it as a client.

Cert-relevant: the difference between
  - Internal MCP (we own and run, exposed via FastMCP) ← internal_mcp.py
  - External MCP (vendor-provided, we configure as client) ← this file
"""
import os


def get_external_mcp_client():
    """Connect to the external CRM MCP server."""
    url = os.environ.get("EXTERNAL_MCP_URL", "https://crm.example.com/mcp")
    api_key = os.environ.get("EXTERNAL_MCP_API_KEY", "demo-key")
    return ExternalMCPHandle(url=url, api_key=api_key)


class ExternalMCPHandle:
    """Lightweight wrapper that the agents will invoke.

    In production this would wrap an actual MCP ClientSession over HTTP/SSE.
    For the demo we mock responses so it runs without a real CRM server.
    """
    def __init__(self, url: str, api_key: str):
        self.url = url
        self.api_key = api_key
        self._connected = True

    async def list_tools(self) -> list[str]:
        """Discover what tools the vendor exposes."""
        return ["crm_search_contacts", "crm_get_account", "crm_log_interaction"]

    async def call_tool(self, name: str, arguments: dict) -> dict:
        """Invoke a tool on the external MCP server."""
        if name == "crm_get_account":
            return {
                "account_id": arguments["account_id"],
                "name": "Maria González",
                "segment": "Premium B2C",
                "satisfaction_score": 8.5,
                "last_contact": "2026-04-14",
                "notes": "Long-time customer. Previous escalation about delivery delay resolved positively.",
            }
        return {"error": "UNKNOWN_TOOL", "tool": name}

    def close(self) -> None:
        self._connected = False

In [ ]:
# File 12 · tools/__init__.py
# Empty — marks tools/ as a Python package

In [ ]:
# File 13 · tools/internal_tools.py
"""Internal tools — plain Python functions registered directly with agents.

Distinction (cert-relevant):
- MCP tools: served by an MCP server, dynamically discovered.
- Native tools: registered locally with one specific agent. Used here for
  in-process operations like grep and HTTP search.
"""
import os
import re
from pathlib import Path
import httpx


def grep_local_file(pattern: str, file_path: str, max_lines: int = 20) -> dict:
    """Search for a regex pattern in a local file and return matching lines."""
    path = Path(file_path)
    if not path.exists():
        return {"status": "error", "message": f"File {file_path} not found"}

    try:
        regex = re.compile(pattern, re.IGNORECASE)
    except re.error as e:
        return {"status": "error", "message": f"Invalid regex: {e}"}

    matches = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if regex.search(line):
                matches.append({"line": line_no, "text": line.rstrip()})
                if len(matches) >= max_lines:
                    break

    return {
        "status": "success",
        "file": file_path,
        "pattern": pattern,
        "match_count": len(matches),
        "matches": matches,
    }


def web_search(query: str, max_results: int = 5) -> dict:
    """Run a web search via an external search API (Brave / Tavily / SerpAPI)."""
    api_key = os.environ.get("SEARCH_API_KEY")
    if not api_key:
        # Mock for the demo so you can run without a real API key
        return {
            "status": "success",
            "query": query,
            "results": [
                {
                    "title": f"Mock result for '{query}'",
                    "url": "https://example.com/mock",
                    "snippet": "Placeholder result. Set SEARCH_API_KEY for real provider.",
                }
            ],
        }

    try:
        with httpx.Client(timeout=10.0) as client:
            r = client.get(
                "https://api.search.brave.com/res/v1/web/search",
                headers={"X-Subscription-Token": api_key},
                params={"q": query, "count": max_results},
            )
            r.raise_for_status()
            data = r.json()
            return {
                "status": "success",
                "query": query,
                "results": [
                    {
                        "title": item["title"],
                        "url": item["url"],
                        "snippet": item.get("description", ""),
                    }
                    for item in data.get("web", {}).get("results", [])[:max_results]
                ],
            }
    except httpx.HTTPError as e:
        return {"status": "error", "message": str(e)}

In [ ]:
# File 14 · subagents/__init__.py
from subagents.base import BaseSubAgent
from subagents.orders import OrdersSubAgent
from subagents.policy import PolicySubAgent
from subagents.crm import CRMSubAgent
from subagents.summarizer import SummarizerAgent

__all__ = [
    "BaseSubAgent",
    "OrdersSubAgent",
    "PolicySubAgent",
    "CRMSubAgent",
    "SummarizerAgent",
]

In [ ]:
# File 15 · subagents/base.py
"""Base class for all subagents.

Every subagent inherits from this to get:
- Access to the Jumpstarter (shared state)
- Lifecycle hook integration (on_tool_call / on_tool_result for audit logging)
- Structured result envelope helpers
- Confidence-based escalation logic
"""
from abc import ABC, abstractmethod
from anthropic import Anthropic

from jumpstarter import Jumpstarter
from models import SubAgentResult, AgentStatus


class BaseSubAgent(ABC):
    """All subagents subclass this. Contract: implement run()."""

    name: str = "base"
    system_prompt: str = ""

    def __init__(self, starter: Jumpstarter):
        self.starter = starter
        self.client = Anthropic(api_key=starter.anthropic_api_key)

    # ──────────────────────────────────────────────────────────
    # Lifecycle hooks (cert: chapter 7, the 7 hooks)
    # ──────────────────────────────────────────────────────────
    def on_tool_call(self, tool_name: str, args: dict) -> None:
        """Audit every tool call before execution. Where rate limiting,
        permission checks, and sensitive-data redaction would also live."""
        self.starter.log_event(
            "tool_call",
            {"agent": self.name, "tool": tool_name, "args": args},
        )

    def on_tool_result(self, tool_name: str, result: dict) -> None:
        """Audit results coming back from tools."""
        self.starter.log_event(
            "tool_result",
            {"agent": self.name, "tool": tool_name, "ok": "error" not in result},
        )

    # ──────────────────────────────────────────────────────────
    # Result envelope helpers — keep them consistent across agents
    # ──────────────────────────────────────────────────────────
    def _success(self, data: dict, confidence: float, tools_used: list[str]) -> SubAgentResult:
        return SubAgentResult(
            agent_name=self.name,
            status=AgentStatus.SUCCESS,
            data=data,
            confidence=confidence,
            tools_used=tools_used,
        )

    def _error(self, msg: str, tools_used: list[str]) -> SubAgentResult:
        return SubAgentResult(
            agent_name=self.name,
            status=AgentStatus.ERROR,
            error_message=msg,
            tools_used=tools_used,
        )

    def _escalate(self, reason: str, partial_data: dict | None, tools_used: list[str]) -> SubAgentResult:
        return SubAgentResult(
            agent_name=self.name,
            status=AgentStatus.ESCALATE,
            data=partial_data,
            escalation_reason=reason,
            confidence=0.0,
            tools_used=tools_used,
        )

    # ──────────────────────────────────────────────────────────
    # Confidence-driven escalation: cert "human escalation triggers"
    # ──────────────────────────────────────────────────────────
    def _check_escalation(self, confidence: float, sensitive: bool) -> str | None:
        """Return an escalation reason string, or None if no escalation needed."""
        if sensitive:
            return "Sensitive topic — human review required"
        if confidence < self.starter.escalation_threshold:
            return f"Low confidence ({confidence:.2f} < {self.starter.escalation_threshold})"
        return None

    @abstractmethod
    def run(self, instruction: str, parameters: dict) -> SubAgentResult:
        """Each subagent implements its own task logic here."""
        raise NotImplementedError

In [ ]:
# File 16 · subagents/orders.py
"""Orders subagent — looks up order details via the internal MCP server.

Includes HITL: state-changing tool calls (returns, refunds, cancellations)
pause for human approval before executing. Read-only calls are not gated.
"""
from subagents.base import BaseSubAgent
from models import SubAgentResult
from hitl import needs_human_approval, ask_human


class OrdersSubAgent(BaseSubAgent):
    name = "orders"
    system_prompt = """You are the Orders specialist agent.

Your job: look up order details using the internal MCP tools available
(search_orders, get_customer_profile, create_return_request).

Return structured data only. If the order doesn't exist or you're unsure,
report it honestly — do not make up information."""

    def _call_tool(self, tool_name: str, args: dict) -> dict:
        """Wraps every MCP tool call with optional HITL approval.

        For read-only tools (search_orders, get_customer_profile) this is
        a passthrough. For sensitive tools (create_return_request, etc.)
        we pause and ask the human before executing.
        """
        # ── HITL gate ─────────────────────────────────────────────
        if needs_human_approval(tool_name):
            self.starter.log_event(
                "hitl_review_requested",
                {"agent": self.name, "tool": tool_name, "args": args},
            )
            if not ask_human(tool_name, args):
                self.starter.log_event(
                    "hitl_denied", {"tool": tool_name, "args": args}
                )
                return {"error": "HUMAN_DENIED", "tool": tool_name}
            self.starter.log_event(
                "hitl_approved", {"tool": tool_name, "args": args}
            )

        # ── Normal tool execution ─────────────────────────────────
        self.on_tool_call(tool_name, args)
        result = self.starter.internal_mcp.call_tool_sync(tool_name, args)
        self.on_tool_result(tool_name, result)
        return result

    def run(self, instruction: str, parameters: dict) -> SubAgentResult:
        tools_used: list[str] = []
        order_id = parameters.get("order_id")
        if not order_id:
            return self._error("Missing order_id parameter", tools_used)

        # Read-only call — no HITL prompt
        result = self._call_tool("search_orders", {"order_id": order_id})
        tools_used.append("internal_mcp.search_orders")

        if "error" in result:
            if result["error"] == "NOT_FOUND":
                return self._escalate(
                    reason=f"Order {order_id} not found — needs human clarification",
                    partial_data={"order_id_searched": order_id},
                    tools_used=tools_used,
                )
            return self._error(result.get("message", "Unknown MCP error"), tools_used)

        # Read-only enrichment — no HITL prompt
        if customer_id := result.get("customer_id"):
            profile = self._call_tool(
                "get_customer_profile", {"customer_id": customer_id}
            )
            tools_used.append("internal_mcp.get_customer_profile")
            result["customer_profile"] = profile

        # ── If the user wants to create a return, this triggers HITL ──
        if parameters.get("action") == "return":
            return_result = self._call_tool(
                "create_return_request",   # ← SENSITIVE — will prompt human
                {"order_id": order_id, "reason": parameters.get("reason", "")},
            )
            tools_used.append("internal_mcp.create_return_request")

            if return_result.get("error") == "HUMAN_DENIED":
                # Human declined — escalate cleanly with partial data
                return self._escalate(
                    reason="Human reviewer denied the return request",
                    partial_data=result,
                    tools_used=tools_used,
                )
            result["return_request"] = return_result

        # ── Write to shared scratchpad so other agents can read it ──
        self.starter.scratchpad["order_details"] = result

        confidence = 0.95
        if reason := self._check_escalation(confidence, sensitive=False):
            return self._escalate(reason, partial_data=result, tools_used=tools_used)

        return self._success(data=result, confidence=confidence, tools_used=tools_used)

In [ ]:
# File 17 · subagents/policy.py
"""Policy subagent — finds the relevant return/refund policy clause.

Cert pattern: uses an INTERNAL TOOL (grep_local_file), not an MCP tool.
Demonstrates that agents can mix MCP tools and native tools.

Also demonstrates the WEB SEARCH tool as a fallback when the local policy
doesn't have an answer.
"""
from subagents.base import BaseSubAgent
from models import SubAgentResult
from tools.internal_tools import grep_local_file, web_search


class PolicySubAgent(BaseSubAgent):
    name = "policy"
    system_prompt = """You are the Policy specialist agent.

Your job: find the exact policy clause that applies to a customer's situation.
Use grep_local_file to search the company policy documents first.
If the local policies don't cover the question, use web_search as a fallback."""

    def run(self, instruction: str, parameters: dict) -> SubAgentResult:
        tools_used: list[str] = []
        topic = parameters.get("topic", "return")

        # ── Step 1: search the local policy file ──
        self.on_tool_call("grep_local_file", {"pattern": topic, "file": "policy.md"})
        local = grep_local_file(
            pattern=topic,
            file_path="docs/return_policy.md",
            max_lines=10,
        )
        self.on_tool_result("grep_local_file", local)
        tools_used.append("grep_local_file")

        if local["status"] == "success" and local["match_count"] > 0:
            self.starter.scratchpad["policy_matches"] = local["matches"]
            return self._success(
                data={"source": "internal_policy", "matches": local["matches"]},
                confidence=0.9,
                tools_used=tools_used,
            )

        # ── Step 2: fall back to web search ──
        self.on_tool_call("web_search", {"query": f"{topic} consumer return policy"})
        web = web_search(query=f"{topic} consumer return policy", max_results=3)
        self.on_tool_result("web_search", web)
        tools_used.append("web_search")

        if web["status"] == "success" and web["results"]:
            confidence = 0.55  # Lower confidence — not our own policy
            if reason := self._check_escalation(confidence, sensitive=False):
                return self._escalate(
                    reason=reason,
                    partial_data={"source": "web_search", "results": web["results"]},
                    tools_used=tools_used,
                )

        return self._error("No policy info found locally or online", tools_used)

In [ ]:
# File 18 · subagents/crm.py
"""CRM subagent — queries the external CRM MCP server.

Cert pattern: uses an EXTERNAL MCP server (vendor-hosted). Tool names are
discovered at runtime from the server, not hardcoded.
"""
import asyncio
from subagents.base import BaseSubAgent
from models import SubAgentResult


class CRMSubAgent(BaseSubAgent):
    name = "crm"
    system_prompt = """You are the CRM specialist agent.

Your job: enrich the case with customer relationship context — segment,
satisfaction history, prior interactions. Use the external CRM MCP tools
(crm_get_account, crm_search_contacts, crm_log_interaction)."""

    def run(self, instruction: str, parameters: dict) -> SubAgentResult:
        tools_used: list[str] = []
        account_id = parameters.get("account_id")

        if not account_id:
            return self._error("Missing account_id parameter", tools_used)

        # ── Discover available tools (real MCP behavior) ──
        try:
            available = asyncio.run(self.starter.external_mcp.list_tools())
            self.on_tool_call("external_mcp.list_tools", {})
            tools_used.append("external_mcp.list_tools")
        except Exception as e:
            return self._error(f"External MCP unreachable: {e}", tools_used)

        if "crm_get_account" not in available:
            return self._error(
                "Required tool 'crm_get_account' not in external MCP", tools_used
            )

        # ── Call the external MCP tool ──
        try:
            self.on_tool_call("crm_get_account", {"account_id": account_id})
            result = asyncio.run(
                self.starter.external_mcp.call_tool(
                    "crm_get_account", {"account_id": account_id}
                )
            )
            self.on_tool_result("crm_get_account", result)
            tools_used.append("external_mcp.crm_get_account")
        except Exception as e:
            return self._error(f"External MCP call failed: {e}", tools_used)

        if "error" in result:
            return self._error(result.get("error", "Unknown CRM error"), tools_used)

        # ── Sensitive-topic check based on CRM data ──
        # Cert pattern: certain attributes trigger automatic escalation,
        # regardless of confidence. E.g. low satisfaction + premium tier.
        sensitive = (
            result.get("satisfaction_score", 10) < 5
            or "complaint" in result.get("notes", "").lower()
        )

        confidence = 0.85
        if reason := self._check_escalation(confidence, sensitive=sensitive):
            return self._escalate(reason, partial_data=result, tools_used=tools_used)

        # ── Stash in scratchpad for the summarizer to read ──
        self.starter.scratchpad["crm_profile"] = result
        return self._success(data=result, confidence=confidence, tools_used=tools_used)

In [ ]:
# File 19 · subagents/summarizer.py
"""Summarizer agent — aggregates the 3 subagents' outputs into a final answer.

Cert pattern: in the Coordinator/Aggregator pattern, the Aggregator's job
is to combine specialist outputs. It does NOT call tools — it just reasons
over the structured data the others produced.
"""
import json
from subagents.base import BaseSubAgent
from models import SubAgentResult, FinalAnswer, AgentStatus


class SummarizerAgent(BaseSubAgent):
    name = "summarizer"
    system_prompt = """You are the Summarizer agent.

You receive structured outputs from three specialist agents (orders, policy,
CRM) and produce a single, coherent customer-facing answer.

Rules:
- Be concise and empathetic.
- Cite which agent each fact came from.
- If any agent escalated, note that in `needs_human_followup`.
- NEVER invent information not present in the agent outputs."""

    def aggregate(self, results: list[SubAgentResult]) -> FinalAnswer:
        """Combine the 3 subagent results into a final answer."""
        successful = [r for r in results if r.status == AgentStatus.SUCCESS]
        escalations = [r for r in results if r.status == AgentStatus.ESCALATE]
        errors = [r for r in results if r.status == AgentStatus.ERROR]

        sources = [r.agent_name for r in successful]

        # Schema-first prompting (cert: structured-output level 3).
        prompt = f"""Customer query: {self.starter.user_query}

Specialist agent outputs (JSON):
{json.dumps([r.model_dump() for r in successful], indent=2, default=str)}

Escalations needing human follow-up:
{json.dumps([r.model_dump() for r in escalations], indent=2, default=str)}

Errors encountered:
{json.dumps([r.model_dump() for r in errors], indent=2, default=str)}

Produce a JSON object with EXACTLY these keys:
{{
  "answer": "<customer-facing answer>",
  "needs_human_followup": <true/false>,
  "followup_reason": "<reason if needs_human_followup is true, else null>"
}}

Return ONLY valid JSON. No preamble, no explanation, no markdown."""

        self.on_tool_call("anthropic.messages.create", {"model": "claude-sonnet-4-5"})
        response = self.client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1024,
            system=self.system_prompt,
            messages=[{"role": "user", "content": prompt}],
        )
        self.on_tool_result("anthropic.messages.create", {"ok": True})

        # ── Parse with retry-on-malformed (cert pattern) ──
        text = response.content[0].text
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            # Retry once with the bad output included in the prompt.
            retry = self.client.messages.create(
                model="claude-sonnet-4-5",
                max_tokens=1024,
                system=self.system_prompt,
                messages=[
                    {"role": "user", "content": prompt},
                    {"role": "assistant", "content": text},
                    {"role": "user", "content": "That was malformed. Return ONLY valid JSON, no other text."},
                ],
            )
            parsed = json.loads(retry.content[0].text)

        return FinalAnswer(
            answer=parsed["answer"],
            sources=sources,
            needs_human_followup=parsed["needs_human_followup"] or len(escalations) > 0,
            followup_reason=parsed.get("followup_reason"),
        )

    def run(self, instruction: str, parameters: dict) -> SubAgentResult:
        # Not used — invoked via aggregate() directly by the coordinator.
        raise NotImplementedError("Use .aggregate(results) instead")

In [ ]:
# File 20 · coordinator.py
"""Coordinator agent — receives the user query, decomposes it into tasks,
dispatches to subagents, handles success/error/escalate, calls the summarizer.

Cert-relevant patterns:
- Coordinator does NOT execute work; it plans and delegates.
- Retry with backoff on transient errors.
- Human escalation when subagents return ESCALATE status.
- Structured task assignments instead of free-text.
"""
import json
import time
from anthropic import Anthropic

from jumpstarter import Jumpstarter
from models import SubAgentResult, AgentStatus, TaskAssignment, FinalAnswer
from subagents.orders import OrdersSubAgent
from subagents.policy import PolicySubAgent
from subagents.crm import CRMSubAgent
from subagents.summarizer import SummarizerAgent


MAX_RETRIES = 3
BACKOFF_BASE_SECONDS = 1.0


class Coordinator:
    """The brain of the system — plans, delegates, retries, escalates."""

    def __init__(self, starter: Jumpstarter):
        self.starter = starter
        self.client = Anthropic(api_key=starter.anthropic_api_key)

        # Instantiate subagents — they all share the Jumpstarter
        self.subagents = {
            "orders": OrdersSubAgent(starter),
            "policy": PolicySubAgent(starter),
            "crm": CRMSubAgent(starter),
        }
        self.summarizer = SummarizerAgent(starter)

    # ──────────────────────────────────────────────────────────
    # 1. DECOMPOSITION — split the user query into structured tasks
    # ──────────────────────────────────────────────────────────
    def decompose_query(self, query: str) -> list[TaskAssignment]:
        """Ask Claude to decompose the user's query into TaskAssignment objects.

        Cert pattern: the coordinator USES Claude for the planning step.
        It doesn't try to parse the query with regex or keyword matching.
        """
        decomposition_prompt = f"""You are the planning component of a customer support system.

You have three specialist sub-agents available:
- "orders"  → looks up order details (param: order_id; optional: action="return", reason)
- "policy"  → finds relevant company policy (param: topic)
- "crm"     → retrieves customer relationship data (param: account_id)

User query: "{query}"

Decompose this query into a JSON array of task assignments. Each task should
target ONE agent. Return ONLY the JSON array, no other text. Example:

[
  {{"target_agent": "orders", "instruction": "Look up the order", "parameters": {{"order_id": "ORD-12345"}}}},
  {{"target_agent": "policy", "instruction": "Find return policy", "parameters": {{"topic": "return"}}}}
]
"""
        response = self.client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1024,
            messages=[{"role": "user", "content": decomposition_prompt}],
        )
        raw = response.content[0].text

        tasks_data = json.loads(raw)
        tasks = [TaskAssignment(**t) for t in tasks_data]

        self.starter.log_event("decomposition", {"task_count": len(tasks)})
        return tasks

    # ──────────────────────────────────────────────────────────
    # 2. DISPATCH — run each task with retry + escalation handling
    # ──────────────────────────────────────────────────────────
    def dispatch_with_retry(self, task: TaskAssignment) -> SubAgentResult:
        """Run a task on a subagent with exponential-backoff retry."""
        agent = self.subagents[task.target_agent]
        last_result: SubAgentResult | None = None

        for attempt in range(MAX_RETRIES):
            result = agent.run(task.instruction, task.parameters)
            last_result = result

            if result.status == AgentStatus.SUCCESS:
                self.starter.log_event(
                    "subagent_success",
                    {"agent": task.target_agent, "attempt": attempt + 1},
                )
                return result

            if result.status == AgentStatus.ESCALATE:
                # Terminal — no point retrying. Pass through to summarizer.
                self.starter.log_event(
                    "subagent_escalate",
                    {"agent": task.target_agent, "reason": result.escalation_reason},
                )
                return result

            # status == ERROR → retry with exponential backoff
            self.starter.increment_retry(task.target_agent)
            backoff = BACKOFF_BASE_SECONDS * (2 ** attempt)
            self.starter.log_event(
                "subagent_retry",
                {
                    "agent": task.target_agent,
                    "attempt": attempt + 1,
                    "error": result.error_message,
                    "backoff_seconds": backoff,
                },
            )
            if attempt < MAX_RETRIES - 1:
                time.sleep(backoff)

        # Retries exhausted → escalate
        self.starter.log_event(
            "subagent_exhausted_retries", {"agent": task.target_agent}
        )
        return SubAgentResult(
            agent_name=task.target_agent,
            status=AgentStatus.ESCALATE,
            escalation_reason=f"All {MAX_RETRIES} retries failed: {last_result.error_message if last_result else 'unknown'}",
            confidence=0.0,
            tools_used=last_result.tools_used if last_result else [],
        )

    # ──────────────────────────────────────────────────────────
    # 3. ORCHESTRATE — the full pipeline end-to-end
    # ──────────────────────────────────────────────────────────
    def orchestrate(self) -> FinalAnswer:
        """Run the full pipeline: decompose → dispatch → summarize."""
        tasks = self.decompose_query(self.starter.user_query)

        results: list[SubAgentResult] = []
        for task in tasks:
            result = self.dispatch_with_retry(task)
            results.append(result)

        # Even if some agents escalated, we summarize what we DO have
        # and flag the escalation in the final answer.
        final = self.summarizer.aggregate(results)

        self.starter.log_event(
            "orchestration_complete",
            {
                "tasks_run": len(tasks),
                "needs_human_followup": final.needs_human_followup,
            },
        )
        return final

In [ ]:
# File 21 · main.py
"""Entry point — runs one query through the full pipeline.

Usage:
    python main.py "Where is my order ORD-12345 and what's the return policy?"
"""
import sys
import json
from rich.console import Console
from rich.panel import Panel
from rich.json import JSON

from jumpstarter import Jumpstarter
from coordinator import Coordinator


def main() -> int:
    console = Console()

    if len(sys.argv) < 2:
        console.print("[red]Usage: python main.py \"<your query>\"[/red]")
        return 1

    query = " ".join(sys.argv[1:])
    console.print(Panel(query, title="User Query", style="cyan"))

    # ── Jumpstarter pattern: ONE state object passed everywhere ──
    starter = Jumpstarter(user_query=query, user_id="user-001")
    starter.bootstrap()

    try:
        coordinator = Coordinator(starter)
        final = coordinator.orchestrate()

        console.print(Panel(final.answer, title="Final Answer", style="green"))
        console.print(f"[dim]Sources: {', '.join(final.sources)}[/dim]")
        if final.needs_human_followup:
            console.print(
                Panel(
                    f"⚠  Human follow-up needed: {final.followup_reason}",
                    style="yellow",
                )
            )

        # ── Show audit log for inspection ──
        console.print("\n[bold]Session audit log:[/bold]")
        console.print(JSON(json.dumps(starter.audit_log, default=str, indent=2)))

    finally:
        starter.shutdown()

    return 0


if __name__ == "__main__":
    sys.exit(main())

# File 22 · docs/return_policy.md
# Customer Return Policy

## General Returns

Returns are accepted within 30 days of delivery for unused items in original
packaging. Refunds are processed to the original payment method within 7
business days of receiving the returned item.

## Damaged or Defective Items

If your item arrives damaged or defective, contact support within 14 days.
A replacement or full refund will be issued, including return shipping costs.
This applies to cracked screens, manufacturing defects, and shipping damage.

## Final Sale Items

Items marked "final sale" cannot be returned or exchanged. This includes
clearance items, perishable goods, and personalized products.

## Premium Tier Customers

Gold and Platinum tier customers receive extended 60-day return windows and
free return shipping on all eligible items.

# File 23 · README.md
# Customer Support Multi-Agent System

A reference implementation showcasing the Anthropic-Certified-Architect patterns:
**Coordinator + 3 specialist subagents + Summarizer**, with **Jumpstarter** for
shared state, **FastMCP** for the internal MCP server, an **external MCP** for
CRM data, and **HITL** on the orders agent.

## Architecture

```
                       ┌──────────────────┐
   User Query ───────► │   Coordinator    │ (decomposes via Claude)
                       └────────┬─────────┘
                                │ TaskAssignments
                ┌───────────────┼───────────────┐
                ▼               ▼               ▼
         ┌──────────┐    ┌──────────┐    ┌──────────┐
         │  Orders  │    │  Policy  │    │   CRM    │
         │ Internal │    │  grep +  │    │ External │
         │   MCP    │    │ web srch │    │   MCP    │
         │(FastMCP) │    │          │    │ (vendor) │
         │  + HITL  │    │          │    │          │
         └────┬─────┘    └────┬─────┘    └────┬─────┘
              │ SubAgentResult: SUCCESS / ERROR / ESCALATE
              └───────────────┼───────────────┘
                              ▼
                       ┌──────────────────┐
                       │   Summarizer     │ → FinalAnswer
                       └──────────────────┘

Shared across ALL boxes: ◄── Jumpstarter (state, MCP clients, audit log)
```

## Setup

```bash
pip install -r requirements.txt
cp .env.example .env
# fill in ANTHROPIC_API_KEY at minimum
```

## Run

```bash
# Read-only query — no HITL prompt
python main.py "Where is my order ORD-12345 and what's the return policy?"

# Query that triggers HITL (return creation)
python main.py "I want to return order ORD-12345 because the screen is cracked"
```

## Run the internal MCP server standalone (optional)

```bash
python -m mcp_servers.internal_mcp
```

## File map

| File | Purpose |
|---|---|
| `.mcp.json` | MCP server registry (project-shared) |
| `.claude/settings.json` | Project-scoped Claude settings + permissions |
| `jumpstarter.py` | Shared session state, MCP clients, audit log |
| `models.py` | Pydantic models — SubAgentResult, TaskAssignment, FinalAnswer |
| `hitl.py` | Minimal HITL helper — used by orders agent only |
| `coordinator.py` | Decomposes query, dispatches, retries, escalates |
| `subagents/base.py` | Common agent scaffolding + lifecycle hooks |
| `subagents/orders.py` | Sub-agent 1 — internal MCP (FastMCP) + HITL |
| `subagents/policy.py` | Sub-agent 2 — grep + web_search internal tools |
| `subagents/crm.py` | Sub-agent 3 — external MCP (vendor) |
| `subagents/summarizer.py` | Aggregator — combines outputs into final answer |
| `mcp_servers/internal_mcp.py` | FastMCP server with tools + resources |
| `mcp_servers/external_mcp_config.py` | Client config for external CRM MCP |
| `tools/internal_tools.py` | grep_local_file + web_search |
| `docs/return_policy.md` | Sample policy doc grep'd by policy agent |
| `main.py` | Entry point |

## Cert-relevant patterns

| Pattern | Where it lives |
|---|---|
| Coordinator pattern | `coordinator.py` — coordinator decomposes & delegates, never executes |
| Single Responsibility per subagent | Each subagent has ONE job |
| MCP Tools vs Resources | `internal_mcp.py` — `search_orders` is a Tool, `order_schema` is a Resource |
| Internal vs External MCP | FastMCP server (we own) vs vendor MCP (we configure as client) |
| Native vs MCP tools | `grep_local_file` and `web_search` are native; MCP calls are dynamic |
| Lifecycle hooks | `on_tool_call` / `on_tool_result` audit-log every invocation |
| Structured agent communication | Pydantic `SubAgentResult` envelope — never free-form text |
| Three terminal states | SUCCESS / ERROR / ESCALATE |
| Retry with exponential backoff | `dispatch_with_retry()` in coordinator |
| Confidence-driven escalation | `_check_escalation()` in `base.py` |
| Sensitive-topic auto-escalation | CRM agent escalates on low satisfaction or "complaint" keywords |
| Retry-with-feedback for malformed JSON | `summarizer.aggregate()` |
| Jumpstarter / shared state | `Jumpstarter` instantiated once, passed by reference everywhere |
| HITL on high-consequence actions | `hitl.py` + `orders.py._call_tool()` |

That's everything. Drop these 23 files into a folder, pip install -r requirements.txt, fill in .env, and python main.py "..." — it runs.

PROMT USED: 
You are a senior solution architect. I need you to generate a complete,
runnable Python project that demonstrates the architectural patterns
from the Anthropic Certified Architect curriculum.

# Domain
Customer support multi-agent system (orders + policy + CRM).

# Required architectural patterns
Generate the project so that EACH of these patterns is clearly identifiable
and commented in the code:

1. **Coordinator pattern** — one coordinator agent that:
   - Receives the user query
   - Uses Claude itself to decompose the query into typed task assignments
   - Dispatches each task to the appropriate specialist subagent
   - Never executes tool calls itself; only plans and delegates

2. **Jumpstarter pattern for shared state** — a single class instantiated
   once per session that holds:
   - Session metadata (id, user, query, timestamp)
   - Configuration loaded from environment
   - MCP client connections (opened in `bootstrap()`, closed in `shutdown()`)
   - Shared scratchpad dict that all agents read/write
   - Audit log of every event
   - Retry counters per subagent
   Pass this object by reference into every subagent — do not duplicate
   state in agent prompts.

3. **Three specialist subagents**, each with single responsibility:
   - Orders agent → uses internal MCP server (FastMCP)
   - Policy agent → uses native Python tools (grep_local_file + web_search)
   - CRM agent → uses external (vendor-hosted) MCP server

4. **Internal MCP server built with FastMCP**:
   - Exposes both Tools (search_orders, get_customer_profile,
     create_return_request) and Resources (order schema)
   - Tools that modify state vs Resources that are read-only — make the
     distinction explicit in comments
   - Standalone runnable: `python -m mcp_servers.internal_mcp`

5. **External MCP server (configured as a client)**:
   - Vendor-hosted, async client interface
   - Tool discovery at runtime via list_tools()
   - Mock the responses so the demo runs without a real vendor server
   - Show the difference vs the internal MCP clearly

6. **Native (non-MCP) internal tools**:
   - grep_local_file using regex on a local markdown file
   - web_search via Brave/Tavily-style HTTP API (mocked when no API key)

7. **Three terminal states for subagent results** (SUCCESS / ERROR / ESCALATE):
   - Use a Pydantic SubAgentResult envelope
   - Coordinator branches:
     - SUCCESS → pass to next stage
     - ERROR → exponential-backoff retry (max 3 attempts)
     - ESCALATE → terminal, pass to summarizer with escalation flag
   - Each subagent decides its own status based on confidence, sensitive
     content, or tool errors

8. **Summarizer agent** that:
   - Aggregates the 3 subagent outputs
   - Uses schema-first prompting (returns JSON only)
   - Implements retry-with-feedback for malformed JSON (the canonical
     "include the bad output in the retry prompt" pattern)
   - Produces a typed FinalAnswer

9. **HITL on the orders subagent only** — minimal, not over-engineered:
   - A small `hitl.py` module with `needs_human_approval(tool_name)` and
     `ask_human(tool_name, args)` (CLI prompt)
   - SENSITIVE_TOOLS set: create_return_request, process_refund, cancel_order
   - In OrdersSubAgent, wrap MCP tool calls in a `_call_tool` method that
     checks the HITL gate before executing
   - Read-only calls (search_orders, get_customer_profile) bypass HITL
   - HUMAN_DENIED is returned as a structured error so the agent can
     escalate cleanly with partial data
   - Audit-log every HITL event in the Jumpstarter

10. **Standard Anthropic configuration files**:
    - `.mcp.json` at project root with both servers (stdio for internal,
      http for external), using ${VAR} substitution for secrets
    - `.claude/settings.json` with `mcpServers.enabled`, `permissions.allow`,
      `permissions.ask` (framework-level HITL for sensitive tools),
      `permissions.deny` (hard blocks)
    - Use the `"//"` convention for inline JSON comments

11. **Lifecycle hooks** in BaseSubAgent:
    - `on_tool_call(tool_name, args)` — audit before execution
    - `on_tool_result(tool_name, result)` — audit after
    - These map to the cert's 7 lifecycle hooks

# Output format

Produce the response as:

1. A project tree at the top showing every file
2. Each file in its own labeled code block, in dependency order
   (requirements → env → config → models → state → tools → MCP servers
   → base agent → specialist agents → summarizer → coordinator → main)
3. Comprehensive comments explaining WHY each piece exists, not just what
   it does — call out which cert chapter / pattern each piece maps to
4. A README.md at the end with:
   - ASCII architecture diagram
   - Setup steps
   - Two run examples (one that triggers HITL, one that doesn't)
   - File map table
   - Cert-relevant patterns table

# Constraints

- Python 3.10+ syntax (use `dict[str, Any]`, `X | None`, etc.)
- Pydantic 2 for typed models
- FastMCP for the internal MCP server
- Anthropic SDK 0.40+ for Claude API calls
- Mock external services (CRM, web search) so the demo runs without
  third-party API keys
- Keep HITL minimal — only the orders agent, only the CLI prompt, only
  ~30 lines in hitl.py plus ~20 lines added to orders.py
- Use rich for pretty CLI output in main.py
- Use a sample return_policy.md in docs/ that the policy agent will grep

# Style

- Triple-quoted module docstrings explaining purpose and cert relevance
- Section dividers with `# ──────────────────────────────────...`
- Type hints on every function signature
- Structured errors (return `{"error": "...", "message": "..."}`),
  never raw exceptions across agent boundaries